In [ ]:
import glob
import os
import sys

import mir_eval
import numpy as np
import soundfile as sf

sys.path.append("../../scripts")
from prepare_pulseit_data import PulseItConverter

In [ ]:
fs = 48000.0

# this assumes a folder `datasets` next to the `libdamp` framework
choralebricks_folder = "../../../datasets/ChoraleBricks/01_AudioAndAnnotations"
output_folder = "../../../datasets/ChoraleBricks/libdamp_all"

found_files = glob.glob(os.path.join(choralebricks_folder, "**/tracks_normalized/*.wav"), recursive=True)
print(len(found_files), "files found.")

os.makedirs(output_folder, exist_ok=True)

In [ ]:
converter = PulseItConverter()

for wav_file in found_files:
    folder = os.path.dirname(wav_file)
    song = folder.split("/")[-2]
    basename = os.path.splitext(os.path.basename(wav_file))[0]
    f0_file = os.path.join(folder, "../annotations", basename + "_f0.csv")
    assert os.path.exists(f0_file), "Corresponding F0 not found."
    print(song, basename)
    if "fho" in basename:
        continue

    f0_filled_file = os.path.join(folder, "../annotations", basename + "_f0_filled.csv")
    f0_orig = np.loadtxt(f0_filled_file, skiprows=1, usecols=(0, 1), delimiter=";")

    x, fs, f0_track, fc_track, gain_track = converter.convert(
        wav_file, target_fs=fs, f0_csv_file=f0_filled_file, f0_col_names=["t", "f0"], separator=";"
    )

    t_smpl = np.arange(len(x)) / fs
    f0_orig_smpl, _ = mir_eval.melody.resample_melody_series(f0_orig[:, 0], f0_orig[:, 1], (f0_orig[:, 1] > 0), t_smpl)

    sf.write(os.path.join(output_folder, f"{song}_{basename}.wav"), x, int(fs))
    np.save(os.path.join(output_folder, f"{song}_{basename}_f0.npy"), f0_track)
    np.save(os.path.join(output_folder, f"{song}_{basename}_f0_orig.npy"), f0_orig_smpl)
    np.save(os.path.join(output_folder, f"{song}_{basename}_fc.npy"), fc_track)
    np.save(os.path.join(output_folder, f"{song}_{basename}_gain.npy"), gain_track)